# Demo 2 — Prompt caching and the invalidation trap

**AI Cost Management and Token Utilization** · Module 2 · ~8 minutes

> Runs top-to-bottom on live API keys. Every cell that spends money prints what it spent.

---

## What this demo lands

1. A cache write costs **1.25×** input; a cache read costs **0.1×**. Break-even is the **2nd call**.
2. The discount is real and immediate — you can see it in the response metadata.
3. **One dynamic token in the wrong place destroys it silently.** Nothing errors. The bill stays high.

This is the highest return-per-minute demo in the course.

In [ ]:
# --- Setup: install + keys -------------------------------------------------
# Colab: this cell installs everything. Local: it is a no-op if already installed.
%pip install -q anthropic openai tiktoken pandas matplotlib 2>/dev/null

import os, getpass

def need(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    return os.environ[var]

# You need at least one. Anthropic is used for the cache-metadata demos because
# it reports cache reads in a separate, easily inspectable bucket.
need('ANTHROPIC_API_KEY')
# need('OPENAI_API_KEY')   # uncomment if you want the OpenAI comparisons
print('keys loaded')

In [ ]:
# --- Verified rate card, September 2026 ------------------------------------
# Sources (checked 5 Sept 2026):
#   platform.claude.com/docs/en/about-claude/pricing
#   developers.openai.com/api/docs/pricing
#   ai.google.dev/gemini-api/docs/pricing
#   deepseek.ai/pricing
# USD per 1,000,000 tokens.  cache_w = 5-minute cache write, cache_r = cache read.

PRICES = {
    # model id                       input  output  cache_w  cache_r
    'deepseek-v4-flash':            dict(inp=0.14, out=0.28,  cw=0.14,  cr=0.0028),
    'gpt-5.6-luna':                 dict(inp=0.20, out=1.20,  cw=0.25,  cr=0.02),
    'gemini-3.5-flash-lite':        dict(inp=0.30, out=2.50,  cw=0.30,  cr=0.03),
    'gemini-3.8-flash':             dict(inp=0.75, out=3.75,  cw=0.75,  cr=0.075),
    'claude-haiku-4-5':             dict(inp=1.00, out=5.00,  cw=1.25,  cr=0.10),
    'claude-sonnet-5':              dict(inp=2.00, out=10.00, cw=2.50,  cr=0.20),
    'gpt-5.6-terra':                dict(inp=2.00, out=12.00, cw=2.50,  cr=0.20),
    'claude-opus-5':                dict(inp=5.00, out=25.00, cw=6.25,  cr=0.50),
    'gpt-5.6-sol':                  dict(inp=4.00, out=20.00, cw=5.00,  cr=0.40),
    'gpt-6-astra':                  dict(inp=10.00,out=50.00, cw=12.50, cr=1.00),
    'claude-fable-5-1':             dict(inp=10.00,out=50.00, cw=12.50, cr=0.25),
}

def cost(model, inp=0, out=0, cache_w=0, cache_r=0):
    """Cost in USD for one call, given token counts by billing bucket."""
    p = PRICES[model]
    return (inp*p['inp'] + out*p['out'] + cache_w*p['cw'] + cache_r*p['cr']) / 1e6

def usd(x):
    return f'${x:,.6f}' if x < 0.01 else f'${x:,.4f}' if x < 1 else f'${x:,.2f}'

print(f'{len(PRICES)} models loaded')

In [ ]:
# --- Cost ledger: every billable call in this notebook lands here ----------
import pandas as pd
LEDGER = []

def log_call(label, model, inp=0, out=0, cache_w=0, cache_r=0, note=''):
    c = cost(model, inp, out, cache_w, cache_r)
    LEDGER.append(dict(label=label, model=model, input=inp, output=out,
                       cache_write=cache_w, cache_read=cache_r, usd=c, note=note))
    print(f'{label:<38} {usd(c):>12}   in={inp:<7} out={out:<6} cw={cache_w:<7} cr={cache_r:<7} {note}')
    return c

def ledger():
    df = pd.DataFrame(LEDGER)
    if df.empty:
        print('no calls yet'); return df
    print(f'\nTOTAL SPENT IN THIS NOTEBOOK: {usd(df.usd.sum())}')
    return df

In [ ]:
import anthropic, time
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'

# A realistic stable prefix: company policy the assistant needs on every call.
# Must exceed the model's minimum cacheable length (1024 tokens for Sonnet/Opus,
# 2048 for Haiku) or caching silently does nothing.
POLICY_BLOCK = ('You are a support assistant for Northwind Logistics.\n\n'
                + '\n'.join(
    f'POLICY {i:03d}: Refund requests under $500 may be approved without escalation when the '
    f'shipment was delayed more than 48 hours and the customer has fewer than three prior claims '
    f'in the trailing twelve months. Document the decision under case note code RF-{i:03d}.'
    for i in range(1, 60)))

print(f'policy block is roughly {len(POLICY_BLOCK)//4:,} tokens')

---
## Call 1 — cold cache (you pay the 1.25× write premium)

In [ ]:
def ask(question, prefix, label):
    r = client.messages.create(
        model=MODEL, max_tokens=120,
        system=[{'type': 'text', 'text': prefix,
                 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user', 'content': question}])
    u = r.usage
    cw = getattr(u, 'cache_creation_input_tokens', 0) or 0
    cr = getattr(u, 'cache_read_input_tokens', 0) or 0
    log_call(label, MODEL, inp=u.input_tokens, out=u.output_tokens,
             cache_w=cw, cache_r=cr,
             note='CACHE HIT' if cr else 'cache write')
    return r, u

r1, u1 = ask('What is the refund threshold?', POLICY_BLOCK, '1. cold cache (write)')
print()
print('raw usage:', u1)

---
## Call 2 — warm cache, different question, identical prefix

Watch `cache_creation_input_tokens` go to 0 and `cache_read_input_tokens` light up.

In [ ]:
time.sleep(1)
r2, u2 = ask('How many prior claims disqualify a customer?', POLICY_BLOCK, '2. warm cache (read)')
print()
print('raw usage:', u2)

---
## The invalidation trap

Now inject a session timestamp **before** the policy block — the single most common
production mistake. The prefix is no longer byte-stable, so the whole cache is invalidated.

In [ ]:
import datetime
BAD_PREFIX = f'Session started: {datetime.datetime.now().isoformat()}\n\n' + POLICY_BLOCK
r3, u3 = ask('What is the refund threshold?', BAD_PREFIX, '3. timestamp AT FRONT')
print('\n^ cache_read collapsed to 0. Full-price write, on every single call.')
print('  Nothing raised an exception. The bill just quietly stayed high.')

### The fix — move the dynamic content to the end

In [ ]:
r4 = client.messages.create(
    model=MODEL, max_tokens=120,
    system=[{'type': 'text', 'text': POLICY_BLOCK,
             'cache_control': {'type': 'ephemeral'}}],
    messages=[{'role': 'user',
               'content': f'What is the refund threshold?\n\n'
                          f'[session {datetime.datetime.now().isoformat()}]'}])
u4 = r4.usage
log_call('4. timestamp AT END', MODEL, inp=u4.input_tokens, out=u4.output_tokens,
         cache_w=getattr(u4,'cache_creation_input_tokens',0) or 0,
         cache_r=getattr(u4,'cache_read_input_tokens',0) or 0,
         note='CACHE HIT' if (getattr(u4,'cache_read_input_tokens',0) or 0) else 'cache write')
print('\n^ the cache read is back. Same information, different position.')

---
## The break-even, derived from the rate card

$$N_{breakeven} = 1 + \frac{P_{write} - P_{input}}{P_{input} - P_{read}}$$

In [ ]:
for m in ['claude-haiku-4-5','claude-sonnet-5','claude-opus-5','claude-fable-5-1']:
    p = PRICES[m]
    n = 1 + (p['cw'] - p['inp']) / (p['inp'] - p['cr'])
    print(f"{m:<22} write={p['cw']:>6.2f}  read={p['cr']:>6.3f}  "
          f"break-even at call {n:.2f} -> call {int(-(-n//1))}")
print('\nThere is essentially no stable-prefix workload where caching loses money.')

---
## The honest caveat: 90% off input is NOT 90% off the bill

Caching only touches input. Output is 5–6× the price of input and is never cached.

In [ ]:
def bill(model, tok_in, tok_out, hit_rate):
    cached = tok_in * hit_rate
    return cost(model, inp=tok_in-cached, out=tok_out, cache_r=cached)

M, TIN, TOUT = 'claude-sonnet-5', 8000, 400
base = bill(M, TIN, TOUT, 0.0)
print(f'{"hit rate":>10} {"cost/call":>12} {"saving":>10}')
for h in [0, .25, .5, .8, .95, 1.0]:
    c = bill(M, TIN, TOUT, h)
    print(f'{h:>9.0%} {usd(c):>12} {1-c/base:>9.0%}')
print('\nAt a realistic 80% hit rate the "90% discount" lands as roughly a third off the total.')
print('Say this out loud before someone in the room does the arithmetic themselves.')

In [ ]:
ledger()

---
## Takeaways

- Caching is a **prompt-ordering discipline**, not a feature flag.
- Static first: system prompt -> tool schemas -> stable policy. Dynamic last: retrieval -> history -> user turn.
- Bad ordering costs **12.5×** more than correct ordering for the same tokens (1.25× write vs 0.1× read).
- Watch `cache_read_input_tokens` in production. If it is near zero, you have this bug.

**Normalisation gotcha for Module 4:** OpenAI counts cached tokens *inside* `prompt_tokens`;
Anthropic reports `cache_read_input_tokens` in a *separate* bucket. Sum them naively and you double-count.